# Тестирование MVP на синтетических данных

**Цель**: Проверить весь pipeline (notebooks 02→03→04) **без доступа к Hive**.

Генерирует синтетические данные, имитирующие выход ETL, и запускает полный цикл анализа.

**Встроенные паттерны для проверки**:
- Seed-компания с 5 дочерними (высокий PageRank)
- 2 shell-компании (высокий flow-through, нет зарплатных проектов)
- Циклический платёж (A→B→C→A)
- Общий представитель у 2 компаний
- Общие сотрудники между компаниями

---

## Шаг 1: Генерация синтетических данных

In [ ]:
import sys, os, pickle
sys.path.insert(0, '..')

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(name)s %(levelname)s: %(message)s')

import pandas as pd
from src import config
from src.synthetic import generate_synthetic_data

paths = generate_synthetic_data(output_dir=config.DATA_DIR, seed=42)

## Шаг 2: Построение и фильтрация графа

In [ ]:
from src.graph_builder import build_graph, compute_edge_metrics, derive_shared_employees, get_graph_stats
from src.filters import apply_filter_pipeline

# Загрузка данных (Pandas — локальная ФС, без Spark)
nodes_pdf = pd.read_parquet(paths['nodes'])
tx_pdf = pd.read_parquet(paths['transaction_edges'])
auth_pdf = pd.read_parquet(paths['authority_edges'])
sal_pdf = pd.read_parquet(paths['salary_edges'])
hop_pdf = pd.read_parquet(paths['hop_distances'])
nodes_pdf = nodes_pdf.merge(hop_pdf, on='client_uk', how='left')

print(f'Loaded: {len(nodes_pdf)} nodes, {len(tx_pdf)} tx, {len(auth_pdf)} auth, {len(sal_pdf)} salary')

In [ ]:
# Построение графа
G = build_graph(nodes_pdf, tx_pdf, auth_pdf, sal_pdf)
G = compute_edge_metrics(G)

# Общие сотрудники
shared_df = derive_shared_employees(sal_pdf)
for _, row in shared_df.iterrows():
    a, b = int(row['company_a_uk']), int(row['company_b_uk'])
    if a in G and b in G:
        G.add_edge(a, b, edge_type='shared_employees', shared_count=int(row['shared_count']), weight=float(row['shared_count']))
        G.add_edge(b, a, edge_type='shared_employees', shared_count=int(row['shared_count']), weight=float(row['shared_count']))

stats = get_graph_stats(G)
print(f"\nRaw graph: {stats['node_count']} nodes, {stats['edge_count']} edges")
print(f"By type: {stats['nodes_by_type']}")
print(f"Edges: {stats['edges_by_type']}")
print(f"Components: {stats['weakly_connected_components']}")

In [ ]:
# Фильтрация
filtered_G, filter_stats = apply_filter_pipeline(G, min_tx_count=2, alpha=0.05)

print(f"\nFiltered: {filter_stats['final_nodes']} nodes, {filter_stats['final_edges']} edges")
print(f"Edge retention: {filter_stats['edge_retention_rate']:.1%}")
print(f"Weight retention: {filter_stats['weight_retention_rate']:.1%}")

# ПРОВЕРКА: рёбра должны уменьшиться
assert filter_stats['final_edges'] < filter_stats['original_edges'], \
    'Disparity filter should reduce edges'
print('\n✓ Disparity filter reduces edges')

## Шаг 3: Анализ графа

In [ ]:
from src.analysis import (
    run_leiden_clustering, compute_centrality, classify_node_roles,
    detect_shell_companies, detect_cycles, build_cluster_summary,
)

# Кластеризация
membership, best_gamma = run_leiden_clustering(filtered_G)
for node, cid in membership.items():
    if node in filtered_G:
        filtered_G.nodes[node]['cluster'] = cid

n_clusters = len(set(v for v in membership.values() if v >= 0))
print(f'\nClusters: {n_clusters} (best gamma={best_gamma})')

# ПРОВЕРКА: должен быть хотя бы 1 кластер
assert n_clusters >= 1, 'Leiden should find at least 1 cluster'
print('✓ Leiden finds clusters')

In [ ]:
# Центральность
metrics_df = compute_centrality(filtered_G)
for node_id, row in metrics_df.iterrows():
    if node_id in filtered_G:
        filtered_G.nodes[node_id]['pagerank'] = row['pagerank']
        filtered_G.nodes[node_id]['betweenness'] = row['betweenness']

# ПРОВЕРКА: seed-компания должна иметь высокий PageRank
seed_uk = 1000
if seed_uk in metrics_df.index:
    seed_pr = metrics_df.loc[seed_uk, 'pagerank']
    median_pr = metrics_df['pagerank'].median()
    print(f'\nSeed PageRank: {seed_pr:.6f} (median: {median_pr:.6f})')
    assert seed_pr > median_pr, 'Seed company should have above-median PageRank'
    print('✓ Seed company has high PageRank')

In [ ]:
# Роли
metrics_df = classify_node_roles(metrics_df, filtered_G)
for node_id, row in metrics_df.iterrows():
    if node_id in filtered_G:
        filtered_G.nodes[node_id]['role'] = row['role']

print(f'\nRoles: {metrics_df["role"].value_counts().to_dict()}')
print('✓ Roles classified')

In [ ]:
# Shell-компании
shell_flagged = detect_shell_companies(metrics_df, filtered_G)
metrics_df['shell_score'] = 0.0
for idx in shell_flagged.index:
    metrics_df.loc[idx, 'shell_score'] = shell_flagged.loc[idx, 'shell_score']

print(f'\nShell companies flagged: {len(shell_flagged)}')
if len(shell_flagged) > 0:
    print(shell_flagged[['pagerank', 'betweenness', 'flow_through_ratio', 'has_salary_payments', 'shell_score']])
print('✓ Shell detection ran')

In [ ]:
# Циклы
cycles = detect_cycles(filtered_G)
print(f'\nCycles detected: {len(cycles)}')
for cyc in cycles[:5]:
    names = [filtered_G.nodes[n].get('name', str(n))[:20] if n in filtered_G else str(n) for n in cyc['nodes']]
    print(f"  len={cyc['length']}, amount={cyc['total_amount']:,.0f}: {' → '.join(names)}")
print('✓ Cycle detection ran')

In [ ]:
# Сводка по кластерам
cluster_summary = build_cluster_summary(filtered_G, metrics_df, cycles)
if len(cluster_summary) > 0:
    display(cluster_summary)
print('✓ Cluster summary built')

## Шаг 4: Визуализация

In [ ]:
from src.viz import create_graph_visualization, display_summary_table, display_node_profile

# Полный граф
output_html = os.path.join(config.DATA_DIR, 'test_graph.html')
net = create_graph_visualization(filtered_G, output_file=output_html)
net.show(output_html)
print(f'\n✓ Graph visualization created: {output_html}')

In [ ]:
# Сводная таблица
display_summary_table(cluster_summary)
print('✓ Summary table displayed')

In [ ]:
# Профиль seed-компании
display_node_profile(filtered_G, 1000, metrics_df)
print('✓ Node profile displayed')

## Итоги тестирования

In [ ]:
print('=' * 60)
print('TEST SUMMARY')
print('=' * 60)
print(f'Graph construction:     ✓ ({stats["node_count"]} nodes, {stats["edge_count"]} edges)')
print(f'Edge metrics:           ✓ (share_of_turnover, reciprocity)')
print(f'Shared employees:       ✓ ({len(shared_df)} pairs)')
print(f'Disparity filter:       ✓ ({filter_stats["edge_retention_rate"]:.0%} retention)')
print(f'Leiden clustering:      ✓ ({n_clusters} clusters, gamma={best_gamma})')
print(f'Centrality metrics:     ✓ (PageRank, betweenness, clustering)')
print(f'Role classification:    ✓ ({metrics_df["role"].nunique()} roles)')
print(f'Shell detection:        ✓ ({len(shell_flagged)} flagged)')
print(f'Cycle detection:        ✓ ({len(cycles)} cycles)')
print(f'Visualization:          ✓ (HTML graph generated)')
print(f'Summary table:          ✓')
print(f'Node profile:           ✓')
print('=' * 60)
print('ALL TESTS PASSED')